# SIPOM — EDA y Entrenamiento
**Sistema Inteligente de Predicción y Optimización de Movilidad Urbana**

Este notebook se ejecuta en Google Colab. Cubre:
1. Carga y exploración (EDA) de los 4 datasets históricos
2. Cruce y construcción del dataset de entrenamiento
3. Entrenamiento del modelo PyTorch y exportación de `sipom_model.pt`

## 0. Instalación de dependencias y montaje de Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib', 'seaborn'], check=True)

print('✅ Dependencias listas')

Mounted at /content/drive
✅ Dependencias listas


In [2]:
import os

RUTA_DATASETS      = '/content/drive/MyDrive/SIPO/Datasets'
RUTA_INCIDENTES    = os.path.join(RUTA_DATASETS, 'incidentes_viales_medellin.csv')
RUTA_PRECIPITACION = os.path.join(RUTA_DATASETS, 'precipitacion_medellin.csv')
RUTA_TEMPERATURA   = os.path.join(RUTA_DATASETS, 'temperatura_medellin.csv')
RUTA_FLUJO         = os.path.join(RUTA_DATASETS, 'flujo_vehicular_medellin.csv')

for nombre, ruta in [
    ('incidentes',    RUTA_INCIDENTES),
    ('precipitación', RUTA_PRECIPITACION),
    ('temperatura',   RUTA_TEMPERATURA),
    ('flujo',         RUTA_FLUJO),
]:
    estado = '✅' if os.path.exists(ruta) else '❌ NO ENCONTRADO'
    print(f'{estado} — {nombre}: {ruta}')

✅ — incidentes: /content/drive/MyDrive/SIPO/Datasets/incidentes_viales_medellin.csv
✅ — precipitación: /content/drive/MyDrive/SIPO/Datasets/precipitacion_medellin.csv
✅ — temperatura: /content/drive/MyDrive/SIPO/Datasets/temperatura_medellin.csv
✅ — flujo: /content/drive/MyDrive/SIPO/Datasets/flujo_vehicular_medellin.csv


## 1. Dataset de Incidentes Viales

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df_inc = pd.read_csv(RUTA_INCIDENTES, encoding='latin-1', low_memory=False)

print(f'Filas: {len(df_inc):,}  |  Columnas: {df_inc.shape[1]}')
df_inc.head(3)

In [ ]:
print('=== TIPOS ===')
print(df_inc.dtypes)
print('\n=== NULOS ===')
print(df_inc.isnull().sum()[df_inc.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_inc['GRAVEDAD_ACCIDENTE'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Incidentes por Gravedad')
axes[0].tick_params(axis='x', rotation=30)

df_inc['NUMCOMUNA'].value_counts().head(10).plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white'
)
axes[1].set_title('Top 10 Comunas con más Incidentes')
axes[1].set_xlabel('NUMCOMUNA')

plt.tight_layout()
plt.show()

In [ ]:
df_inc['fecha_dt'] = pd.to_datetime(
    df_inc['FECHA_ACCIDENTE'], format='%d/%m/%Y %H:%M:%S', errors='coerce'
)

nulos_antes = df_inc['fecha_dt'].isnull().sum()
if nulos_antes > len(df_inc) * 0.1:
    print(f'⚠️  {nulos_antes} fechas no parseadas — usando FECHA_ACCIDENTES')
    df_inc['fecha_dt'] = pd.to_datetime(df_inc['FECHA_ACCIDENTES'], errors='coerce')

df_inc['hora'] = df_inc['fecha_dt'].dt.hour
df_inc['mes']  = df_inc['fecha_dt'].dt.month

print(f'Fechas OK:    {df_inc["fecha_dt"].notna().sum():,}')
print(f'Fechas nulas: {df_inc["fecha_dt"].isna().sum():,}')

df_inc['hora'].value_counts().sort_index().plot(
    kind='bar', figsize=(12, 3), color='teal', edgecolor='white'
)
plt.title('Incidentes por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Cantidad')
plt.tight_layout()
plt.show()

## 2. Dataset de Precipitación

In [ ]:
df_prec = pd.read_csv(RUTA_PRECIPITACION, encoding='latin-1', low_memory=False)

print(f'Filas: {len(df_prec):,}  |  Columnas: {df_prec.shape[1]}')
df_prec.head(3)

In [ ]:
df_prec['ValorObservado'] = (
    df_prec['ValorObservado']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

df_prec['fecha_dt'] = pd.to_datetime(
    df_prec['FechaObservacion'], format='%Y %b %d %I:%M:%S %p', errors='coerce'
)

df_prec_h = (
    df_prec.set_index('fecha_dt')['ValorObservado']
    .resample('h')
    .sum()
    .reset_index()
    .rename(columns={'ValorObservado': 'mm_lluvia', 'fecha_dt': 'fecha_hora'})
)

df_prec_h['lluvia'] = (df_prec_h['mm_lluvia'] > 0.5).astype(int)

print(f'Registros horarios: {len(df_prec_h):,}')
print(f'Horas con lluvia:   {df_prec_h["lluvia"].sum():,}')

## 3. Dataset de Temperatura

In [ ]:
df_temp = pd.read_csv(RUTA_TEMPERATURA, encoding='latin-1', low_memory=False)

df_temp['ValorObservado'] = (
    df_temp['ValorObservado']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

df_temp['fecha_dt'] = pd.to_datetime(
    df_temp['FechaObservacion'], format='%Y %b %d %I:%M:%S %p', errors='coerce'
)

df_temp_h = (
    df_temp.set_index('fecha_dt')['ValorObservado']
    .resample('h')
    .mean()
    .reset_index()
    .rename(columns={'ValorObservado': 'temperatura', 'fecha_dt': 'fecha_hora'})
)

print(f'Registros horarios: {len(df_temp_h):,}')
print(f'Temp mín: {df_temp_h["temperatura"].min():.1f}°C')
print(f'Temp máx: {df_temp_h["temperatura"].max():.1f}°C')

## 4. Dataset de Flujo Vehicular

In [ ]:
df_flujo = pd.read_csv(RUTA_FLUJO, encoding='utf-8', low_memory=False)

print(f'Filas: {len(df_flujo):,}  |  Columnas: {df_flujo.shape[1]}')

df_flujo['fecha_hora'] = pd.to_datetime(
    df_flujo['fecha'].astype(str) + ' ' + df_flujo['hora'].astype(str),
    errors='coerce'
)

df_flujo['hora_int']       = df_flujo['fecha_hora'].dt.hour
df_flujo['dia_semana_int'] = df_flujo['fecha_hora'].dt.dayofweek
df_flujo['mes_int']        = df_flujo['fecha_hora'].dt.month

print(f'Rango: {df_flujo["fecha_hora"].min()} → {df_flujo["fecha_hora"].max()}')
df_flujo.head(3)

## 5. Cruce de los 4 datasets

In [ ]:
df_clima_h = pd.merge(df_temp_h, df_prec_h, on='fecha_hora', how='outer')

df_clima_h['fecha_hora'] = df_clima_h['fecha_hora'].dt.floor('h')

mediana_temp = df_clima_h['temperatura'].median()
df_clima_h['temperatura'] = df_clima_h['temperatura'].fillna(mediana_temp)
df_clima_h['lluvia']      = df_clima_h['lluvia'].fillna(0).astype(int)

print(f'Registros clima por hora: {len(df_clima_h):,}')
df_clima_h.head(3)

In [ ]:
MAPA_COMUNA_ZONA = {
    4:  'Bello',
    5:  'Bello',
    10: 'La Macarena',
    11: 'Laureles',
    12: 'Av. 80',
    13: 'Av. 80',
    14: 'El Poblado',
    15: 'Av. Regional',
    16: 'Av. Regional',
}

df_inc['zona']       = df_inc['NUMCOMUNA'].map(MAPA_COMUNA_ZONA)
df_inc['fecha_hora'] = df_inc['fecha_dt'].dt.floor('h')

df_inc_zh = (
    df_inc
    .dropna(subset=['zona', 'fecha_hora'])
    .groupby(['zona', 'fecha_hora'])
    .size()
    .reset_index(name='incidentes_hora')
)

print(f'Registros incidentes por zona-hora: {len(df_inc_zh):,}')
df_inc_zh.head(5)

In [ ]:
df_flujo['fecha_hora'] = df_flujo['fecha_hora'].dt.floor('h')

df_flujo_base = df_flujo[[
    'fecha_hora', 'zona', 'hora_int', 'dia_semana_int',
    'mes_int', 'velocidad_kmh', 'volumen_vehiculos', 'flujo_promedio'
]].copy()

print(f'Registros flujo base: {len(df_flujo_base):,}')
df_flujo_base.head(3)

In [ ]:
df_merged = pd.merge(df_flujo_base, df_clima_h, on='fecha_hora', how='left')
df_merged = pd.merge(df_merged,     df_inc_zh,  on=['zona', 'fecha_hora'], how='left')

df_merged['incidentes_hora'] = df_merged['incidentes_hora'].fillna(0).astype(int)
df_merged['temperatura']     = df_merged['temperatura'].fillna(mediana_temp)
df_merged['lluvia']          = df_merged['lluvia'].fillna(0).astype(int)

print(f'Filas en dataset cruzado: {len(df_merged):,}')
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])
df_merged.head(4)

## 6. Target y encoding

In [ ]:
MAPA_TARGET = {
    'bajo':     0,
    'medio':    1,
    'alto':     2,
    'muy_alto': 2,
}
df_merged['target'] = df_merged['flujo_promedio'].map(MAPA_TARGET)

nulos_target = df_merged['target'].isnull().sum()
if nulos_target > 0:
    print(f'⚠️  {nulos_target} filas sin target')
    print(df_merged['flujo_promedio'].unique())
else:
    print('✅ Target generado sin nulos')

print(df_merged['target'].value_counts().rename({0:'fluido', 1:'moderado', 2:'crítico'}))

In [ ]:
ZONAS_SIPOM = [
    'Av. 80', 'Av. Regional', 'Calle 33', 'El Poblado', 'Guayabal',
    'Laureles', 'Estadio/Atanasio', 'La Macarena', 'Itagüí', 'Bello',
]
MAPA_ZONA_ID = {zona: idx for idx, zona in enumerate(ZONAS_SIPOM)}

df_merged['zona_id'] = df_merged['zona'].map(MAPA_ZONA_ID)

zonas_sin_id = df_merged[df_merged['zona_id'].isnull()]['zona'].unique()
if len(zonas_sin_id) > 0:
    print(f'⚠️  Zonas no reconocidas: {zonas_sin_id}')
else:
    print('✅ Todas las zonas tienen zona_id')

for zona, idx in MAPA_ZONA_ID.items():
    print(f'  {idx} → {zona}')

In [ ]:
FEATURES = [
    'zona_id', 'hora_int', 'dia_semana_int', 'mes_int',
    'temperatura', 'lluvia', 'velocidad_kmh', 'volumen_vehiculos', 'incidentes_hora',
]

df_train = df_merged[FEATURES + ['target']].dropna().reset_index(drop=True)

df_train['zona_id']         = df_train['zona_id'].astype(int)
df_train['hora_int']        = df_train['hora_int'].astype(int)
df_train['dia_semana_int']  = df_train['dia_semana_int'].astype(int)
df_train['mes_int']         = df_train['mes_int'].astype(int)
df_train['lluvia']          = df_train['lluvia'].astype(int)
df_train['incidentes_hora'] = df_train['incidentes_hora'].astype(int)
df_train['target']          = df_train['target'].astype(int)

print(f'Filas para entrenamiento: {len(df_train):,}')
df_train.head()

## 7. Modelo PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

In [ ]:
class SIPOMModel(nn.Module):
    def __init__(self, n_zonas=10, emb_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(n_zonas, emb_dim)
        entrada = emb_dim + 8
        self.red = nn.Sequential(
            nn.Linear(entrada, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, zona_id, features_num):
        emb = self.embedding(zona_id)
        x   = torch.cat([emb, features_num], dim=1)
        return self.red(x)

modelo = SIPOMModel().to(DEVICE)
total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {total_params:,}')

## 8. Dataset y DataLoaders

In [ ]:
class SIPOMDataset(Dataset):
    def __init__(self, df):
        self.zona_id  = torch.tensor(df['zona_id'].values, dtype=torch.long)
        cols_num      = ['hora_int', 'dia_semana_int', 'mes_int',
                         'temperatura', 'lluvia', 'velocidad_kmh',
                         'volumen_vehiculos', 'incidentes_hora']
        self.features = torch.tensor(df[cols_num].values, dtype=torch.float32)
        self.target   = torch.tensor(df['target'].values, dtype=torch.long)

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return self.zona_id[idx], self.features[idx], self.target[idx]

## 9. Entrenamiento

In [ ]:
df_temp_split, df_test_split = train_test_split(
    df_train, test_size=0.15, random_state=42, stratify=df_train['target']
)
df_tr, df_val = train_test_split(
    df_temp_split, test_size=0.176, random_state=42, stratify=df_temp_split['target']
)

print(f'Train:      {len(df_tr):,} filas')
print(f'Validación: {len(df_val):,} filas')
print(f'Test:       {len(df_test_split):,} filas')

BATCH_SIZE = 256

dl_train = DataLoader(SIPOMDataset(df_tr),         batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
dl_val   = DataLoader(SIPOMDataset(df_val),        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
dl_test  = DataLoader(SIPOMDataset(df_test_split), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Batches por época (train): {len(dl_train)}')

In [ ]:
EPOCHS      = 30
LR          = 1e-3
criterio    = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LR)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizador, mode='min', patience=5, factor=0.5, verbose=True
)

def ejecutar_epoca(modelo, loader, criterio, optimizador=None):
    entrenando    = optimizador is not None
    modelo.train() if entrenando else modelo.eval()
    perdida_total = 0
    correctos     = 0
    total         = 0

    ctx = torch.enable_grad() if entrenando else torch.no_grad()
    with ctx:
        for zona_id, features, target in loader:
            zona_id  = zona_id.to(DEVICE)
            features = features.to(DEVICE)
            target   = target.to(DEVICE)
            salida   = modelo(zona_id, features)
            perdida  = criterio(salida, target)

            if entrenando:
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

            perdida_total += perdida.item() * len(target)
            correctos     += (salida.argmax(dim=1) == target).sum().item()
            total         += len(target)

    return perdida_total / total, correctos / total

In [ ]:
RUTA_MEJOR_MODELO = '/content/drive/MyDrive/SIPO/sipom_model.pt'

historial      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
mejor_val_loss = float('inf')

print(f'Entrenando {EPOCHS} épocas en {DEVICE}...\n')

for epoca in range(1, EPOCHS + 1):
    tr_loss, tr_acc = ejecutar_epoca(modelo, dl_train, criterio, optimizador)
    vl_loss, vl_acc = ejecutar_epoca(modelo, dl_val,   criterio)

    scheduler.step(vl_loss)

    historial['train_loss'].append(tr_loss)
    historial['val_loss'].append(vl_loss)
    historial['train_acc'].append(tr_acc)
    historial['val_acc'].append(vl_acc)

    if vl_loss < mejor_val_loss:
        mejor_val_loss = vl_loss
        torch.save(modelo.state_dict(), RUTA_MEJOR_MODELO)
        marca = ' ← mejor'
    else:
        marca = ''

    print(
        f'Época {epoca:02d}/{EPOCHS} | '
        f'Train loss: {tr_loss:.4f} acc: {tr_acc:.3f} | '
        f'Val loss: {vl_loss:.4f} acc: {vl_acc:.3f}{marca}'
    )

print(f'\n✅ Listo. Mejor val_loss: {mejor_val_loss:.4f}')

## 10. Métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epocas = range(1, EPOCHS + 1)

axes[0].plot(epocas, historial['train_loss'], label='Train')
axes[0].plot(epocas, historial['val_loss'],   label='Validación')
axes[0].set_title('Pérdida por Época')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(epocas, historial['train_acc'], label='Train')
axes[1].plot(epocas, historial['val_acc'],   label='Validación')
axes[1].set_title('Accuracy por Época')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
modelo.load_state_dict(torch.load(RUTA_MEJOR_MODELO, map_location=DEVICE))
modelo.eval()

todas_preds  = []
todos_reales = []

with torch.no_grad():
    for zona_id, features, target in dl_test:
        zona_id  = zona_id.to(DEVICE)
        features = features.to(DEVICE)
        preds    = modelo(zona_id, features).argmax(dim=1).cpu().numpy()
        todas_preds.extend(preds)
        todos_reales.extend(target.numpy())

NOMBRES_CLASES = ['Fluido', 'Moderado', 'Crítico']
print(classification_report(todos_reales, todas_preds, target_names=NOMBRES_CLASES))

In [ ]:
cm = confusion_matrix(todos_reales, todas_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=NOMBRES_CLASES, yticklabels=NOMBRES_CLASES)
plt.title('Matriz de Confusión — Test Set')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## 11. Exportación

In [ ]:
from google.colab import files
import json

# Descargar modelo
files.download(RUTA_MEJOR_MODELO)

# Guardar y descargar mapa de zonas
RUTA_MAPA_ZONAS = '/content/drive/MyDrive/SIPO/zona_id_map.json'
with open(RUTA_MAPA_ZONAS, 'w', encoding='utf-8') as f:
    json.dump(MAPA_ZONA_ID, f, ensure_ascii=False, indent=2)

files.download(RUTA_MAPA_ZONAS)

print('✅ Descarga lista')
print('→ Pegar sipom_model.pt   en SIPO/model/')
print('→ Pegar zona_id_map.json en SIPO/model/')